## 1、流式调用 、非流式调用

In [1]:
# 非流式调用
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, SystemMessage, HumanMessage
import dotenv
from openai import max_retries
from sympy.physics.units import temperature

dotenv.load_dotenv()
# 1、获取大模型的实例
model = init_chat_model(
    model="deepseek-chat",
    model_provider="deepseek",
    temperature = 0.5,
)
res = model.invoke("你好，你是谁")
print(res)

content='你好！我是DeepSeek，由深度求索公司创造的AI助手！😊\n\n我是一个纯文本模型，虽然不支持多模态识别功能，但我有文件上传功能，可以帮你处理图像、txt、pdf、ppt、word、excel等文件，并从中读取文字信息进行分析处理。我完全免费使用，拥有128K的上下文长度，还支持联网搜索功能（需要你在Web/App中手动点开联网搜索按键）。\n\n你可以通过官方应用商店下载我的App来使用。我很乐意为你解答问题、协助处理各种任务，或者就是简单地聊聊天！\n\n有什么我可以帮助你的吗？无论是学习、工作还是生活中的问题，我都很愿意为你提供帮助！✨' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 7, 'total_tokens': 151, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 7}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache', 'id': 'f91cd34b-a791-4bad-90c9-9eb16b498343', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d70dd-f284-7ea3-9bdd-123a7f2843a5-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 7, 'output_tokens': 144, 'total_tokens': 151, 'input_token_details'

In [2]:
# 流式调用，通过model.stream方法去调用，
# 返回的是一个生成器，通过迭代生成器的方式，得到结果
res = model.stream("你好，你是谁")

In [3]:
for chunk in res:
    print(chunk.content,end="")

你好！我是DeepSeek，由深度求索公司创造的AI助手！😊

我是一个纯文本模型，虽然不支持多模态识别功能，但我有文件上传功能，可以帮你处理图像、txt、pdf、ppt、word、excel等文件，并从中读取文字信息进行分析处理。我完全免费使用，拥有128K的上下文长度，还支持联网搜索功能（需要你在Web/App中手动点开联网搜索按键）。

你可以通过官方应用商店下载我的App来使用。我很乐意为你解答问题、协助处理各种任务，或者就是简单地聊聊天！有什么我可以帮助你的吗？✨

## 2、批次调用、非批次调用

In [4]:
# 批次调用，通过model.batch方法实现，底层原理就是通过多线程的方式去调用，
#
messages = [
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于春天的诗"},
    ],
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于夏天的诗"},
    ],
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于秋天的诗"},
    ],
]
res = model.batch(messages)

## 3、同步调用、异步调用

In [5]:
# 同步调用：多次请求之间串行处理，B请求需要A请求完成之后，再发出请求，得到响应
messagess = [
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于春天的诗"},
    ],
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于夏天的诗"},
    ],
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于秋天的诗"},
    ],
]
import time
start_time = time.time()
res = [model.invoke(messages) for messages in messagess]
end_time = time.time()
print(f"总耗时:{end_time - start_time}")

总耗时:15.34364366531372


In [6]:
# 异步调用：model.ainvoke方法，返回一个协程对象，把多个协程对象可以打包成一个协程对象，
# await最终的协程对象，就能够实现异步调用
# 异步调用，能够提高程序的性能。相对于batch调用而言，能够减少资源（线程数）使用量
import asyncio
async def gather_task(messages:list):
    # 调用ainvoke并不会真正地发起请求
    tasks = [model.ainvoke(message_list) for message_list in messages]
    return await asyncio.gather(*tasks)
gather_task(messagess)

<coroutine object gather_task at 0x000001D17F2C17D0>

In [7]:
await gather_task(messagess)

[AIMessage(content='《春讯》\n东君昨夜过江城，袖底风回万象更。\n冻柳垂金试新水，老梅抱雪落残英。\n泥融南圃芹芽短，日上西墙杏子明。\n最喜初雷惊蛰后，邻家已有饭牛声。\n\n注：我的诗以“春讯”为题，通过冻柳垂金、老梅落英等意象展现冬春交替的细腻过程。后两联转写田园生机，泥融芹短、初雷饭牛等场景，以农耕文明最朴素的节律呼应天地律动。全诗试图在古典意象中注入生活温度，让春讯不止于物候变化，更成为大地血脉苏醒的脉搏。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 156, 'prompt_tokens': 12, 'total_tokens': 168, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 12}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache', 'id': '48c46222-9e75-4f39-93fa-8d8d0219febf', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d70de-607e-7fd0-b009-b73c3a769ab9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 156, 'total_tokens': 168, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}),
 AIM